<a href="https://colab.research.google.com/github/Kotoal/housing/blob/main/CV_HousePricing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# =======================
# IMPORTS
# =======================
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt

# =======================
# LOAD DATA
# =======================
data = pd.read_csv("Housing.csv")

# =======================
# CLEANING
# =======================
data = data.copy()

for col in data.select_dtypes(include='object').columns:
    data[col] = data[col].str.lower().str.strip()

# yes/no → 1/0
for col in data.columns:
    if data[col].dtype == 'object' and col != 'furnishingstatus':
        data[col] = data[col].map({'yes': 1, 'no': 0})

# one-hot
data = pd.get_dummies(data, columns=['furnishingstatus'])

# =======================
# FEATURE ENGINEERING
# =======================
data['room_density'] = data['area'] / (data['bedrooms'] + 1)
data['bath_ratio'] = data['bathrooms'] / (data['bedrooms'] + 1)
data['comfort'] = data[['airconditioning', 'prefarea']].sum(axis=1)
data['house_score'] = data['stories'] * data['parking']

# =======================
# SPLIT
# =======================
X = data.drop('price', axis=1).values
y = data['price'].values.reshape(-1,1)

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.25)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5)

# =======================
# NORMALIZATION
# =======================
sc_X = StandardScaler()
sc_y = StandardScaler()

X_train = sc_X.fit_transform(X_train)
X_val = sc_X.transform(X_val)
X_test = sc_X.transform(X_test)

y_train = sc_y.fit_transform(y_train)
y_val = sc_y.transform(y_val)
y_test = sc_y.transform(y_test)

# =======================
# TENSORS
# =======================
train_data = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32)
)

train_loader = DataLoader(train_data, batch_size=16, shuffle=True)

X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)

# =======================
# MODEL
# =======================
class PriceNet(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.LeakyReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, 64),
            nn.LeakyReLU(),

            nn.Linear(64, 16),
            nn.ReLU(),

            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.model(x)

net = PriceNet(X_train.shape[1])

# =======================
# TRAIN
# =======================
optimizer = torch.optim.Adam(net.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

train_hist = []
val_hist = []

for epoch in range(180):
    net.train()
    batch_loss = 0

    for xb, yb in train_loader:
        optimizer.zero_grad()
        out = net(xb)
        loss = loss_fn(out, yb)
        loss.backward()
        optimizer.step()
        batch_loss += loss.item()

    train_loss = batch_loss / len(train_loader)

    net.eval()
    with torch.no_grad():
        val_pred = net(X_val_t)
        val_loss = loss_fn(val_pred, y_val_t).item()

    train_hist.append(train_loss)
    val_hist.append(val_loss)

    if epoch % 30 == 0:
        print(f"Epoch {epoch}: train={train_loss:.4f}, val={val_loss:.4f}")

# =======================
# GRAPH
# =======================
plt.plot(train_hist, label="train")
plt.plot(val_hist, label="val")
plt.legend()
plt.title("Loss curve")
plt.show()

# =======================
# EVALUATION
# =======================
net.eval()
with torch.no_grad():
    pred = net(X_test_t).numpy()

pred = sc_y.inverse_transform(pred)
true = sc_y.inverse_transform(y_test)

rmse = np.sqrt(mean_squared_error(true, pred))
r2 = r2_score(true, pred)

print("\nRESULTS")
print("RMSE:", rmse)
print("R2:", r2)

# =======================
# SAVE
# =======================
torch.save(net.state_dict(), "price_model_v2.pth")

# =======================
# LOAD + TEST
# =======================
loaded = PriceNet(X_train.shape[1])
loaded.load_state_dict(torch.load("price_model_v2.pth"))
loaded.eval()

sample = X_test_t[0].unsqueeze(0)

with torch.no_grad():
    p = loaded(sample).numpy()

p = sc_y.inverse_transform(p)

print("\nPrediction:", p[0][0])
print("Actual:", true[0][0])

FileNotFoundError: [Errno 2] No such file or directory: 'Housing.csv'